# 01. Data, datums, and the temporal framework

We start with two lidar DTMs of the focal region. One was flown in September 2013. The other was flown in fall 2022, six to twelve weeks after the McKinney Fire and the August 2 storm. By the end of this block both epochs are in GRASS, registered in a space-time raster dataset, and differenced into a first DEM of Difference (DoD). The rest of the workshop is about how far to trust it.

In [ ]:
import io
import os
import subprocess

from workshop_utils import RAW, gj, start

tools, session = start("01")

## Two files, one label

Both files report the same coordinate reference system (CRS).

In [ ]:
epochs = {
    "dtm_2013": RAW / "usfs_2013_dtm.tif",
    "dtm_2022": RAW / "usgs_2022_dtm.tif",
}
for name, path in epochs.items():
    crs = tools.g_proj(flags="p", georef=str(path), format="projjson")
    print(f"{name}: {crs['name']}")

The collection metadata tells a different story.

| Epoch | Horizontal datum | Vertical datum |
| --- | --- | --- |
| 2013, USFS | NAD83 | NAVD88, GEOID12A |
| 2022, USGS 3DEP | NAD83(2011) | NAVD88, GEOID18 |

The file headers do not record the NAD83 realization. If we tell PROJ the 2013 file is the original NAD83 and ask for NAD83(2011), it applies the NADCON5 grid shift. Without the grid it silently applies none.

In [ ]:
point = "522000 4625000 0\n"
for network in ("OFF", "ON"):
    out = subprocess.run(
        ["cs2cs", "-d", "3", "EPSG:26910", "EPSG:6339"],
        input=point,
        capture_output=True,
        text=True,
        env={**os.environ, "PROJ_NETWORK": network},
    ).stdout.split()
    print(f"PROJ_NETWORK={network}: 522000 4625000 -> {out[0]} {out[1]}")

With the grid, that point moves about 0.6 m. The geoid models differ too. Here GEOID18 sits 2 to 3 cm below GEOID12A, so the same ground reads 2 to 3 cm higher in 2022 before anything has changed.

We do not resolve either offset by relabeling files. The project uses EPSG:26910, the CRS both files declare, so nothing is reprojected on import and the result does not depend on whether PROJ can reach its grids. Block 02 measures the remaining horizontal and vertical offset on stable ground instead.

## Import both epochs

The computational region is the focal area at 1 m. `r.import` clips each file to the region and resamples it onto the same 1 m grid. The resampling matters here. The 2022 grid starts on whole meters, but the 2013 grid starts at 520422.29, 4626838.83, so its cells sit 0.71 m and 0.83 m off the region grid. That is a gridding difference, not a co-registration offset, and bilinear resampling handles it.

In [ ]:
for name, path in epochs.items():
    tools.r_import(
        input=str(path),
        output=name,
        extent="region",
        resolution="region",
        resample="bilinear",
        overwrite=True,
    )
    tools.r_relief(input=name, output=f"{name}_relief", overwrite=True)

In [ ]:
fire = gj.Map(width=700)
fire.d_shade(shade="dtm_2022_relief", color="dtm_2022")
fire.d_vect(map="mckinney_perimeter@PERMANENT", fill_color="none", color="red", width=2)
fire.d_legend(
    raster="dtm_2022", title="Elevation 2022 (m)", at=(5, 40, 2, 5), flags="b"
)
fire.show()

The red line is the MTBS fire perimeter. The fire control line follows Humbug Creek, so part of this region did not burn. That unburned ground is what we use to measure error.

## The temporal framework

With two epochs a time series sounds like overkill. It stops being overkill the first time you add a third collection, and it keeps the acquisition dates attached to the data. Each epoch is registered with its collection window as a time interval.

In [ ]:
with open("epochs.txt", "w") as f:
    f.write("dtm_2013|2013-09-04|2013-09-08\n")
    f.write("dtm_2022|2022-09-14|2022-10-29\n")

tools.t_create(
    output="dtm",
    type="strds",
    temporaltype="absolute",
    title="Humbug Creek lidar DTMs",
    description="Bare-earth DTMs, 1 m, focal region",
    overwrite=True,
)
tools.t_register(input="dtm", type="raster", file="epochs.txt", overwrite=True)
print(tools.t_rast_list(input="dtm", columns="name,start_time,end_time").text)

## A naive DoD

Subtract the earlier epoch from the later one, pulling the map names from the dataset in time order. Negative values are surface lowering (erosion), positive values are surface raising (deposition).

In [ ]:
names = tools.t_rast_list(
    input="dtm", columns="name", order="start_time", flags="u"
).text.split()
before, after = names[0], names[-1]
tools.r_mapcalc(expression=f"dod_naive = {after} - {before}", overwrite=True)
# Blue for lowering, red for raising, saturated at +/-1 m.
tools.r_colors(
    map="dod_naive",
    rules=io.StringIO("-1000 blue\n-1 blue\n0 white\n1 red\n1000 red\n"),
)
print(f"dod_naive = {after} - {before}")

In [ ]:
dod = gj.Map(width=700)
dod.d_shade(shade="dtm_2022_relief", color="dod_naive")
dod.d_vect(
    map="mckinney_perimeter@PERMANENT", fill_color="none", color="black", width=2
)
dod.d_legend(
    raster="dod_naive", title="DoD (m)", range=(-1, 1), at=(5, 40, 2, 5), flags="b"
)
dod.show()

Two patterns stand out. Humbug Creek and the side channels are red, which is deposition after the storm. On the hillslopes, opposite-facing slopes lean slightly toward opposite colors. A bias that follows aspect is what a horizontal offset between the epochs looks like.

## Burned and unburned ground

If the DoD were all real change, unburned ground would average close to zero. Compare the two.

In [ ]:
tools.v_to_rast(
    input="mckinney_perimeter@PERMANENT",
    output="burned_area",
    use="val",
    value=1,
    overwrite=True,
)
tools.r_mapcalc(expression="burned = if(isnull(burned_area), 0, 1)", overwrite=True)
tools.r_category(
    map="burned", separator=":", rules=io.StringIO("0:unburned\n1:burned\n")
)

stats = tools.r_univar(map="dod_naive", zones="burned", flags="e", format="json")
for zone in stats:
    print(
        f"{zone['zone_label']:>9}: mean {zone['mean']:+.3f} m, median {zone['median']:+.3f} m, "
        f"std {zone['stddev']:.3f} m, cells {zone['n']}"
    )

Write down the unburned mean and standard deviation. A nonzero mean on ground that did not burn is systematic error, which includes the datum offsets above. The spread is noise. Block 02 removes the first and block 03 turns the second into a level of detection.

## Your turn

Error on steep slopes is often larger because a small horizontal offset turns into a large vertical one. Split the unburned ground into slope classes and compare the DoD spread. Change the class breaks and see how the numbers move.

In [ ]:
tools.r_slope_aspect(elevation="dtm_2013", slope="slope_2013", overwrite=True)

# Slope classes in degrees. Try other breaks.
tools.r_mapcalc(
    expression=(
        "slope_class = if(burned == 0, "
        "if(slope_2013 < 15, 1, if(slope_2013 < 30, 2, 3)), null())"
    ),
    overwrite=True,
)
tools.r_category(
    map="slope_class",
    separator=":",
    rules=io.StringIO("1:0-15 deg\n2:15-30 deg\n3:30+ deg\n"),
)

for zone in tools.r_univar(map="dod_naive", zones="slope_class", format="json"):
    print(
        f"{zone['zone_label']:>10}: mean {zone['mean']:+.3f} m, std {zone['stddev']:.3f} m"
    )

Map the mean offset of each slope class back onto the unburned ground. If the offset were a single vertical shift, every class would get the same color. Rerun this cell after changing the class breaks.

In [ ]:
tools.r_stats_zonal(
    base="slope_class",
    cover="dod_naive",
    method="average",
    output="slope_offset",
    overwrite=True,
)
tools.r_colors(
    map="slope_offset",
    rules=io.StringIO("-1000 blue\n-0.2 blue\n0 white\n0.2 red\n1000 red\n"),
)

# Zoom the display to the unburned ground, then restore the computational region.
tools.g_region(zoom="slope_offset", save="unburned_view", overwrite=True)
tools.g_region(region="focal@PERMANENT")

offsets = gj.Map(width=700, saved_region="unburned_view")
offsets.d_rast(map="dtm_2013_relief")
offsets.d_shade(shade="dtm_2013_relief", color="slope_offset")
offsets.d_vect(
    map="mckinney_perimeter@PERMANENT", fill_color="none", color="black", width=2
)
offsets.d_legend(
    raster="slope_offset",
    title="Mean offset (m)",
    range=(-0.2, 0.2),
    at=(65, 95, 3, 7),
    flags="b",
)
offsets.show()

With the default breaks, gentle slopes sit near +0.13 m and slopes steeper than 30 degrees near +0.06 m. A vertical shift alone would give the same offset everywhere, so part of this error depends on the terrain. Block 02 separates the vertical shift from the horizontal one.